# Latent spatial probability-flow atlas — proof of concept

**Question:** does a molecular probability flow learned from one healthy PT realization predict the next spatial molecular distribution in the other?

For unordered cross-sections, posit a canonical position $z_i\in[0,1]$, conditional molecular distributions $\rho_z^{(s)}(x)$, and a shared velocity $v(x,z)$ satisfying the idealized continuity equation $\partial_z\rho_z+\nabla_x\cdot(\rho_zv)=0$. We infer velocity directly; drift and diffusion are not separately estimated. Position indexes anatomy, not elapsed time or a tracked tubule.

This notebook starts with **Ctrl1A2 and Ctrl1A4 only**, current PT DPT as temporary scaffolding, and marker-excluded low-dimensional expression. Experiment 1 must predict held-out distributions better than identity, centroid displacement, and DPT-local mean displacement in **both specimen directions**, including a non-overlapping-window sensitivity. Only then run the latent-position pilot and atlas-only ablation. AKI projection and a TF feasibility check are gated further.

The implemented approximation distills adjacent entropic-OT barycentric displacements into a smooth affine field, then evaluates integrated pushforwards by multivariate energy distance. It does not reproduce PFI or optimize the full joint inverse problem. A Gaussian atlas blends empirical moments with one-step flow-predicted moments; alternating posterior reassignment is a pilot, not exact EM or an enforced PDE solution.

With two controls this is a descriptive go/no-go experiment. Overlapping windows share structures, DPT carries upstream molecular information, and weak anatomy inherits annotation evidence. Withheld markers do not erase that indirect information. No exact longitudinal location is observed; a monotone coordinate reparameterization remains unidentifiable. The uniform atlas grid is an explicit gauge, not physical length or a sampling-density estimate.

The eventual joint inverse problem would combine
$\mathcal L_{\rm data}+\lambda_F\mathcal L_{\rm flow}+\lambda_A\mathcal L_{\rm anatomy}+\lambda_S\mathcal L_{\rm specimen}+\lambda_R\mathcal R(v)$:
conditional density fit, predictive pushforward consistency, weak anatomical order,
a shared main positional program with restricted specimen effects, and field smoothness.
The present fixed-axis experiment tests flow consistency first. The latent pilot
uses a Gaussian approximation and empirical/transported moment blending; it does
not claim to optimize that entire objective or identify specimen effects.


In [ ]:
from pathlib import Path
import argparse, hashlib, json, os, sys
import numpy as np
import pandas as pd
import anndata as ad
from scipy import sparse
from scipy.spatial import cKDTree
from scipy.stats import spearmanr, rankdata
from scipy.special import logsumexp
import matplotlib.pyplot as plt
from IPython.display import display
REPO = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / 'pseudospace').is_dir())
sys.path.insert(0, str(REPO))
from pseudospace.repeated_inputs import load_repeated_mouse_inputs, _read_and_validate_segmentation
from pseudospace.spatial_probability_flow import scaffold_centroid_rows, representation, window_clouds, fit_flow, advect, benchmark, latent_fit, posterior
from pseudospace.stage_cache import cached_frame, cached_payload
parser = argparse.ArgumentParser(add_help=False)
parser.add_argument('--data-root', default=os.environ.get('PSEUDOSPACE_DATA_ROOT', str(REPO / 'data')))
parser.add_argument('--results-root', default=os.environ.get('PSEUDOSPACE_RESULTS_ROOT', str(REPO / 'results')))
args, _ = parser.parse_known_args()
DATA_ROOT, RESULTS_ROOT = Path(args.data_root).expanduser(), Path(args.results_root).expanduser()
DPT_PATH = Path(os.environ.get('PSEUDOSPACE_15_DPT_REFERENCE', str(RESULTS_ROOT / 'human_vs_healthy_mouse' / 'cross_species_pt_dpt.h5ad')))
DPT_COLUMN = os.environ.get('PSEUDOSPACE_15_DPT_COLUMN', 'total_scanpy_dpt')
GLOMERULUS_PATH = Path(os.environ.get('PSEUDOSPACE_15_GLOMERULI', str(RESULTS_ROOT / 'human_vs_healthy_mouse' / 'cross_species_harmony_pass1.h5ad')))
OUT = RESULTS_ROOT / 'spatial_probability_flow_proof_of_concept'
OUT.mkdir(parents=True, exist_ok=True)
CONTROLS = ('Ctrl1A2', 'Ctrl1A4')
NOTEBOOK_LOGIC_VERSION = '15.flow.3'
SEED, N_WINDOWS, N_COMPONENTS = 15, 12, 15
MIN_COUNT, MAX_POINTS = 12, 128
RIDGE, EPSILON, ITERATIONS = 1., .5, 4
N_GENE_FOLDS = 3
N_BOOT = int(os.environ.get('PSEUDOSPACE_15_BOOTSTRAPS', '10'))
RUN_AKI = os.environ.get('PSEUDOSPACE_15_AKI', '0') == '1'
RUN_TF = os.environ.get('PSEUDOSPACE_15_TF', '0') == '1'
if N_BOOT < 2:
    raise ValueError('At least two bootstrap repetitions are required')
MARKERS = set('Slc5a2 Slc5a12 Gatm Slc22a6 Slc13a3 Slc1a1 Cyp2e1 Slc7a13 Slc22a7 Slc5a1 Cyp7b1 Slc7a12 Lrp2 Cubn Slc34a1 Slc6a18 Acsm3 Agt'.casefold().split())
CODE_FILES = [REPO / 'pseudospace' / f for f in ('spatial_probability_flow.py', 'repeated_inputs.py')]
CODE_DIGEST = hashlib.sha256(b''.join(p.read_bytes() for p in CODE_FILES)).hexdigest()
CACHE_PARAMS = dict(version=NOTEBOOK_LOGIC_VERSION, seed=SEED, windows=N_WINDOWS, components=N_COMPONENTS, min_count=MIN_COUNT, max_points=MAX_POINTS, ridge=RIDGE, epsilon=EPSILON, iterations=ITERATIONS)
plt.rcParams.update({'font.size': 9, 'axes.spines.top': False, 'axes.spines.right': False, 'pdf.fonttype': 42, 'svg.fonttype': 'none'})

## 1. Healthy measured PT and fixed DPT scaffold

The loader checks current fine-segmentation feature IDs and centroids, applies the pooled **healthy-cohort** mouse fifth-percentile spot-count floor, and normalizes across common measured genes before PT selection. It applies no tubule gene-count QC. AKI inputs are unnecessary here. The changed floor cohort is recorded and can differ from 02's four-mouse floor.

The default scaffold is 03's saved PT DPT, already oriented upstream. A mouse-only export can be supplied with `PSEUDOSPACE_15_DPT_REFERENCE` and, if needed, `PSEUDOSPACE_15_DPT_COLUMN=pt_subset_scanpy_dpt`. This notebook never reorients DPT using held-out markers. The centroid join must be unique within one pixel in each specimen, and feature IDs must agree when exported. Structures absent from the supplied PT DPT are explicitly reported as scaffold-ineligible; this fixed-axis experiment uses the intersection with validated current PT. This can reflect differing upstream label/filter contracts. No missing coordinate is imputed. Expression need not match because only the saved coordinate is borrowed. Its cross-species origin and feature leakage are recorded. Global healthy min/max scaling preserves the common axis; specimens are never rescaled separately.

In [ ]:
healthy = load_repeated_mouse_inputs(DATA_ROOT, samples=CONTROLS)
obs = healthy.obs.copy()
genes = healthy.var_names.to_numpy(str)
ids = obs.structure_id.to_numpy(str)
specimens = obs['sample'].astype(str).to_numpy()
labels = obs.segment_class.astype(str).str.replace('PT-', '', regex=False).to_numpy()
if not np.isin(labels, ['S1', 'S2', 'S3']).all():
    raise ValueError('Resolve uncertain PT labels upstream; do not silently drop structures')
M = healthy.layers['lognorm']
X = M.toarray().astype(float) if sparse.issparse(M) else np.asarray(M, float)
if not np.isfinite(X).all():
    raise ValueError('Expression contains nonfinite values')
if not DPT_PATH.is_file():
    raise FileNotFoundError(f'Run the existing reviewed DPT workflow first, or supply PSEUDOSPACE_15_DPT_REFERENCE: {DPT_PATH}')
reference = ad.read_h5ad(DPT_PATH)
required = {'sample', 'x_centroid', 'y_centroid', DPT_COLUMN}
if not required.issubset(reference.obs):
    raise ValueError(f'DPT reference missing {required - set(reference.obs)}')
# Some existing exports omit feature_index. Use a strictly unique centroid match.
xy = obs[['x_centroid', 'y_centroid']].to_numpy(float)
row = scaffold_centroid_rows(xy, specimens, reference.obs[['x_centroid', 'y_centroid']].to_numpy(float), reference.obs['sample'].astype(str).to_numpy(), tolerance=1.)
if 'feature_index' in reference.obs:
    present = row >= 0
    ref_feature = pd.to_numeric(reference.obs.iloc[row[present]].feature_index, errors='raise').to_numpy(float)
    source_feature = obs.iloc[np.flatnonzero(present)].feature_index.to_numpy(float)
    if not np.array_equal(ref_feature, source_feature):
        raise ValueError('DPT feature indices disagree with validated centroid join')
# A fixed-axis experiment can only include structures carrying the saved DPT.
# Report this scaffold eligibility explicitly; it is not a gene-count QC filter.
eligible = row >= 0
excluded_scaffold = obs.loc[~eligible, ['structure_id', 'sample', 'segment_class']].copy()
excluded_scaffold['reason'] = 'absent from supplied PT DPT scaffold'
excluded_scaffold.to_csv(OUT / 'scaffold_ineligible_structures.csv', index=False)
display(pd.crosstab(specimens, eligible, colnames=['DPT eligible']))
healthy = healthy[eligible].copy()
obs = obs.loc[eligible].copy()
X, ids, specimens, labels, xy, row = X[eligible], ids[eligible], specimens[eligible], labels[eligible], xy[eligible], row[eligible]
z_raw = reference.obs[DPT_COLUMN].to_numpy(float)[row]
if not np.isfinite(z_raw).all() or np.ptp(z_raw) <= 0:
    raise ValueError('DPT must be finite and nonconstant')
z_dpt = (z_raw-z_raw.min()) / np.ptp(z_raw)
marker_mask = np.array([g.casefold() in MARKERS for g in genes])
gene_fold = np.array([int(hashlib.sha256(g.casefold().encode()).hexdigest()[:8], 16) % N_GENE_FOLDS for g in genes])
model_genes = np.flatnonzero(~marker_mask & (gene_fold != 0))
eval_genes = np.flatnonzero(~marker_mask & (gene_fold == 0))
if min(len(model_genes), len(eval_genes)) < N_COMPONENTS:
    raise ValueError('Insufficient marker-excluded modeling/evaluation genes')
lo = max(z_dpt[specimens == s].min() for s in CONTROLS)
hi = min(z_dpt[specimens == s].max() for s in CONTROLS)
if hi <= lo:
    raise ValueError('No common DPT support')
centers = np.linspace(lo, hi, N_WINDOWS)
spacing = centers[1]-centers[0]
widths = {'overlapping': .8*spacing, 'nonoverlap': .499*spacing}
manifest = dict(healthy.uns['repeated_mouse_input_manifest'])
manifest.update(scaffold_eligible_structures=len(X), scaffold_excluded_structures=len(excluded_scaffold), scaffold_join='unique within-specimen centroid within 1 pixel; feature_index checked when available', scaffold_path=str(DPT_PATH), scaffold_column=DPT_COLUMN, scaffold_sha256=hashlib.sha256(z_raw.tobytes()).hexdigest(), dpt_scaled_healthy_range=[float(z_raw.min()), float(z_raw.max())], common_support=[float(lo), float(hi)], markers_excluded=sorted(MARKERS), model_genes=genes[model_genes].tolist(), evaluation_genes=genes[eval_genes].tolist(), config=CACHE_PARAMS, code_digest=CODE_DIGEST)
(OUT / 'input_manifest.json').write_text(json.dumps(manifest, indent=2))
display(pd.crosstab(specimens, labels))
print('Dense matrix GiB:', X.nbytes/2**30, 'DPT scaffolding:', DPT_PATH.name)

## 2. Visualize the distribution path and window support

Each canonical window gives equal weight to the two specimens in the pooled visualization; the actual benchmark fits **one control** and evaluates the other. PCA/scaling is fitted on that training control alone. Display individual cloud points and specimen means, rather than hiding heterogeneity behind a principal curve. The first plot is a training-representation diagnostic, not evidence of generalization.

In [ ]:
train0 = specimens == CONTROLS[0]
YA, YB, transform0 = representation(X[train0][:, model_genes], X[~train0][:, model_genes], N_COMPONENTS, SEED)
Y_view = np.empty((len(X), YA.shape[1]))
Y_view[train0], Y_view[~train0] = YA, YB
support_rows = []
for scheme, halfwidth in widths.items():
    for s in CONTROLS:
        for k, center in enumerate(centers):
            support_rows.append(dict(scheme=scheme, specimen=s, window=k, center=center, n=int(np.sum((specimens==s) & (np.abs(z_dpt-center)<=halfwidth)))))
support = pd.DataFrame(support_rows)
support.to_csv(OUT / 'window_support.csv', index=False)
display(support.groupby(['scheme', 'specimen']).n.agg(['min', 'max']))
fig, ax = plt.subplots(figsize=(6, 4), layout='constrained')
for s, color in zip(CONTROLS, ['#0072B2', '#D55E00']):
    take = specimens == s
    cloud = window_clouds(Y_view[take], z_dpt[take], centers, widths['overlapping'], MIN_COUNT, MAX_POINTS, SEED)
    means = np.array([a.mean(axis=0) for a in cloud])
    ax.plot(means[:, 0], means[:, 1], 'o-', color=color, label=s)
    for a in cloud:
        ax.scatter(a[:, 0], a[:, 1], s=3, alpha=.08, color=color)
ax.set(xlabel='Training-only PC1', ylabel='Training-only PC2', title='Healthy PT distribution path on DPT scaffold')
ax.legend()
fig.savefig(OUT / 'distribution_path.pdf')
plt.show()

## 3. Experiment 1: cross-specimen molecular transport

For training clouds $W_k$, balanced entropic OT supplies finite-step barycentric displacements. Ridge regression fits $v(y,z)=[1,z,z^2,y,zy]B$. RK4 integrates that field over each adjacent-window step. OT entropy is scaled to each pair's median positive squared molecular distance; all settings are fixed before evaluating the other specimen. This ceiling is deliberately simple: affine fields cannot resolve multimodal state-dependent branching, and barycentric transport may contract variance. A failure here rejects this version, not every possible probability-flow model.

Compare the same held-out source/target clouds using identity, training centroid displacement, local kNN mean displacement along training DPT, and learned flow. Lower multivariate energy distance is better. Adjacent test windows are used **only for scoring**. The source distribution is observed in the test specimen; this tests one-step prediction conditional on its supplied DPT window, not de novo coordinate inference.

Overlapping windows can reward small shifts and share observations. The non-overlap sensitivity must also pass. No test-based hyperparameter tuning, significance test, or treating tubules as independent biological replicates.

In [ ]:
def run_fixed_benchmark():
    frames = []
    for train_specimen in CONTROLS:
        tr = specimens == train_specimen
        A, B, _ = representation(X[tr][:, model_genes], X[~tr][:, model_genes], N_COMPONENTS, SEED)
        for scheme, halfwidth in widths.items():
            frame = benchmark(A, z_dpt[tr], B, z_dpt[~tr], centers, halfwidth, min_count=MIN_COUNT, max_points=MAX_POINTS, seed=SEED, ridge=RIDGE, epsilon=EPSILON)
            frame['train_specimen'] = train_specimen
            frame['test_specimen'] = CONTROLS[1] if train_specimen == CONTROLS[0] else CONTROLS[0]
            frame['scheme'] = scheme
            frames.append(frame)
    return pd.concat(frames, ignore_index=True)
fixed_scores = cached_frame('fixed_dpt_flow', run_fixed_benchmark, root=OUT / 'stage_cache', params=CACHE_PARAMS, inputs={'X': X[:, model_genes], 'z': z_dpt, 'specimens': specimens, 'centers': centers, 'widths': widths}, code=CODE_DIGEST)
fixed_scores.to_csv(OUT / 'cross_specimen_distribution_prediction.csv', index=False)
fixed_summary = fixed_scores.groupby(['scheme', 'train_specimen', 'method']).energy_distance.mean().unstack('method')
display(fixed_summary)
flow_gate = bool((fixed_summary['flow'] < fixed_summary[['identity', 'centroid', 'dpt_local']].min(axis=1)).all())
(OUT / 'experiment1_gate.json').write_text(json.dumps({'pass': flow_gate, 'rule': 'flow strictly beats all three baselines in both directions and both window schemes', 'descriptive_only': True}, indent=2))
print('Experiment 1 go/no-go:', 'GO to latent pilot' if flow_gate else 'STOP: this flow version has not passed')
fig, ax = plt.subplots(figsize=(6, 4), layout='constrained')
for method in ['identity', 'centroid', 'dpt_local', 'flow']:
    a = fixed_scores.query("scheme == 'nonoverlap' and method == @method")
    ax.plot(a.groupby('z_to').energy_distance.mean(), marker='o', label=method)
ax.set(xlabel='Target DPT window', ylabel='Energy distance (lower is better)', title='Held-out transport: mean of two specimen directions')
ax.legend()
fig.savefig(OUT / 'cross_specimen_transport.pdf')
plt.show()

## 4. Experiment 2: latent atlas alone versus flow-smoothed atlas

Only a passing Experiment 1 enables this pilot. Both models use local diagonal Gaussian distributions on the same grid, variance floors, uniform position prior, and a weak ordinal label likelihood with **learned ordered transition cutpoints**. No fixed S1/S2/S3 thirds or expected marker curves. Model A uses empirical atlas moments; Model B blends them equally with the previous distribution's one-step pushforward moments, then refits after posterior reassignment. This is a flow-smoothed density approximation; it does not guarantee continuity or monotone objective descent.

Training alternates atlas estimation and $E[Z|x,C]$ reassignment. Keep the posterior and entropy. Frozen held-out projection uses **molecular features only**, with no test DPT or labels. Repeat DPT, anatomy-oriented PCA1, and random ordinal initialization; test endpoint-only and unsupervised anatomy settings. Failed sparse-support fits remain failures. Parameters and DPT scaffolding are transductive; independent held-out molecular features and curves evaluate their added value. No unrestricted specimen offsets are fitted: an unseen specimen's shift is penalized instead of being absorbed by remapping.

Rotate three deterministic gene folds before PCA. Infer from modeling genes and score unseen genes using training-only kernel-smoothed gene curves, scaled by training-gene variance. DPT is a supplied-coordinate comparator, not a prospectively mapped method. Label-only ordinal initialization is also scored to check whether anatomy accounts for the apparent gain.

Endpoint-only PCA/random starts omit S2 information; no-anatomy PCA/random starts omit all labels. DPT starts retain upstream marker/label information in every ablation. Total-count normalization also uses the common measured-gene denominator, so gene holdout does not remove compositional dependence.

In [ ]:
def initializations(Y, z, labs, seed, anchor='ordinal'):
    rng = np.random.default_rng(seed)
    pc = rankdata(Y[:, 0]) / (len(Y)+1)
    if anchor == 'none':
        ordinal_score = rng.random(len(labs))
    elif anchor == 'endpoint_only':
        # Unknown middle labels play no role: unlabeled rows can occur anywhere.
        endpoint = np.isin(labs, ['S1', 'S3'])
        endpoint_rank = (labs[endpoint] == 'S3').astype(int)
        if spearmanr(pc[endpoint], endpoint_rank).statistic < 0:
            pc = 1-pc
        ordinal_score = 3*rng.random(len(labs))
        ordinal_score[labs == 'S1'] = rng.random(np.sum(labs == 'S1'))
        ordinal_score[labs == 'S3'] = 2+rng.random(np.sum(labs == 'S3'))
    else:
        ranks = np.array([{'S1': 0, 'S2': 1, 'S3': 2}[a] for a in labs])
        if spearmanr(pc, ranks).statistic < 0:
            pc = 1-pc
        ordinal_score = ranks+rng.uniform(0, .9, len(ranks))
    ordinal = rankdata(ordinal_score) / (len(labs)+1)
    return {'DPT': z.copy(), 'PCA1': lo+(hi-lo)*pc, 'random_ordinal': lo+(hi-lo)*ordinal}

def predict_genes(train_values, train_z, query_z):
    weights = np.exp(-.5*((query_z[:, None]-train_z[None, :]) / widths['overlapping'])**2)
    weights /= np.maximum(weights.sum(axis=1, keepdims=True), 1e-300)
    return weights @ train_values

def gene_score(train_values, train_z, test_values, query_z):
    prediction = predict_genes(train_values, train_z, query_z)
    variance = np.maximum(train_values.var(axis=0), 1e-3)
    return float(np.mean((test_values-prediction)**2 / variance))

def run_latent_benchmark():
    score_rows, coord_rows, failures = [], [], []
    for fold in range(N_GENE_FOLDS):
        mg = np.flatnonzero(~marker_mask & (gene_fold != fold))
        eg = np.flatnonzero(~marker_mask & (gene_fold == fold))
        for train_specimen in CONTROLS:
            tr = specimens == train_specimen
            A, B, _ = representation(X[tr][:, mg], X[~tr][:, mg], N_COMPONENTS, SEED)
            init = initializations(A, z_dpt[tr], labels[tr], SEED)
            for init_name, initial_z in init.items():
                for anchor in ['ordinal', 'endpoint_only', 'none']:
                    initial_z = initializations(A, z_dpt[tr], labels[tr], SEED, anchor=anchor)[init_name]
                    for method, weight in [('atlas', 0.), ('atlas_flow', .5)]:
                        key = dict(fold=fold, train_specimen=train_specimen, initialization=init_name, anchor=anchor, method=method)
                        try:
                            result = latent_fit(A, initial_z, labels[tr], centers, widths['overlapping'], flow_weight=weight, iterations=ITERATIONS, endpoint_only=anchor=='endpoint_only', anatomy_strength=0. if anchor=='none' else .25, min_count=MIN_COUNT, max_points=MAX_POINTS, seed=SEED, ridge=RIDGE, epsilon=EPSILON)
                            projected_z, probability, entropy = posterior(result['atlas'], B)
                            mse = gene_score(X[tr][:, eg], result['z'], X[~tr][:, eg], projected_z)
                            score_rows.append(dict(**key, gene_mse=mse))
                            for j, sid in enumerate(ids[~tr]):
                                sd = np.sqrt(np.sum(probability[j]*(centers-projected_z[j])**2))
                                coord_rows.append(dict(**key, structure_id=sid, z=projected_z[j], posterior_sd=sd, entropy=entropy[j], **{f'p_{k:02d}': float(p) for k, p in enumerate(probability[j])}))
                        except (ValueError, RuntimeError, np.linalg.LinAlgError) as exc:
                            failures.append(dict(**key, reason=str(exc)))
            for name, train_z, test_z in [('DPT', z_dpt[tr], z_dpt[~tr]), ('anatomy_only', init['random_ordinal'], initializations(B, z_dpt[~tr], labels[~tr], SEED)['random_ordinal'])]:
                score_rows.append(dict(fold=fold, train_specimen=train_specimen, initialization='supplied', anchor='supplied', method=name, gene_mse=gene_score(X[tr][:, eg], train_z, X[~tr][:, eg], test_z)))
    return {'scores': pd.DataFrame(score_rows).to_json(orient='table'), 'coordinates': pd.DataFrame(coord_rows).to_json(orient='table'), 'failures': pd.DataFrame(failures).to_json(orient='table')}
latent_scores, latent_coordinates, latent_failures = pd.DataFrame(), pd.DataFrame(), pd.DataFrame()
if flow_gate:
    import io
    latent_payload = cached_payload('latent_gene_folds', run_latent_benchmark, root=OUT / 'stage_cache', params={**CACHE_PARAMS, 'gene_folds': N_GENE_FOLDS}, inputs={'ids': ids, 'X': X, 'z': z_dpt, 'labels': labels, 'specimens': specimens, 'gene_fold': gene_fold, 'markers': marker_mask, 'centers': centers, 'widths': widths}, code=CODE_DIGEST)
    latent_scores = pd.read_json(io.StringIO(str(latent_payload['scores'])), orient='table')
    latent_coordinates = pd.read_json(io.StringIO(str(latent_payload['coordinates'])), orient='table')
    latent_failures = pd.read_json(io.StringIO(str(latent_payload['failures'])), orient='table')
    display(latent_scores)
    display(latent_failures)
else:
    print('Latent-position fitting skipped by Experiment 1 gate')
latent_scores.to_csv(OUT / 'latent_heldout_gene_scores.csv', index=False)
latent_coordinates.to_csv(OUT / 'latent_heldout_coordinates.csv', index=False)
latent_failures.to_csv(OUT / 'latent_fit_failures.csv', index=False)

## 5. Initialization, sparse sampling, and bootstrap stability

Compare projected coordinates across initializations without choosing the best by test scores. Bootstrap training tubules, reconstruction genes, and both; fit on 50% of training structures as a sparse-sampling sensitivity. Each replicate refits scaler, PCA, atlas, and flow. Project the unchanged other control. Report failures and per-structure position standard deviation rather than dropping failed fits. Posterior uncertainty is conditional on a fitted Gaussian model and uniform grid prior; bootstrap variation measures a different uncertainty source. Neither is a calibrated credible interval.

In [ ]:
stability = pd.DataFrame()
initialization_agreement = pd.DataFrame()
if flow_gate and not latent_coordinates.empty:
    agreement_rows = []
    for key, group in latent_coordinates.groupby(['fold', 'train_specimen', 'anchor', 'method']):
        pivot = group.pivot(index='structure_id', columns='initialization', values='z')
        for a, b in [('DPT', 'PCA1'), ('DPT', 'random_ordinal')]:
            if {a, b}.issubset(pivot):
                agreement_rows.append(dict(fold=key[0], train_specimen=key[1], anchor=key[2], method=key[3], comparison=a+' vs '+b, spearman=spearmanr(pivot[a], pivot[b]).statistic, mean_absolute_difference=np.mean(np.abs(pivot[a]-pivot[b]))))
    initialization_agreement = pd.DataFrame(agreement_rows)
    display(initialization_agreement)
initialization_agreement.to_csv(OUT / 'initialization_agreement.csv', index=False)

def run_stability():
    rng = np.random.default_rng(SEED)
    rows = []
    for train_specimen in CONTROLS:
        tr = specimens == train_specimen
        base_rows = np.flatnonzero(tr)
        test_rows = np.flatnonzero(~tr)
        for scheme in ['tubules', 'genes', 'both', 'sparse_half']:
            for repeat in range(N_BOOT):
                rr = rng.choice(base_rows, len(base_rows), replace=True) if scheme in ['tubules', 'both'] else base_rows
                if scheme == 'sparse_half':
                    rr = rng.choice(base_rows, max(2, len(base_rows)//2), replace=False)
                gg = rng.choice(model_genes, len(model_genes), replace=True) if scheme in ['genes', 'both'] else model_genes
                A, B, _ = representation(X[rr][:, gg], X[test_rows][:, gg], N_COMPONENTS, SEED)
                for method, weight in [('atlas', 0.), ('atlas_flow', .5)]:
                    try:
                        result = latent_fit(A, z_dpt[rr], labels[rr], centers, widths['overlapping'], flow_weight=weight, iterations=ITERATIONS, min_count=MIN_COUNT, max_points=MAX_POINTS, seed=SEED, ridge=RIDGE, epsilon=EPSILON)
                        zz, _, _ = posterior(result['atlas'], B)
                        for sid, value in zip(ids[test_rows], zz):
                            rows.append(dict(train_specimen=train_specimen, scheme=scheme, repeat=repeat, method=method, structure_id=sid, z=value, failure=''))
                    except (ValueError, RuntimeError, np.linalg.LinAlgError) as exc:
                        rows.append(dict(train_specimen=train_specimen, scheme=scheme, repeat=repeat, method=method, structure_id='', z=np.nan, failure=str(exc)))
    return pd.DataFrame(rows)
if flow_gate:
    stability = cached_frame('latent_stability', run_stability, root=OUT / 'stage_cache', params={**CACHE_PARAMS, 'n_boot': N_BOOT}, inputs={'ids': ids, 'X': X, 'z': z_dpt, 'labels': labels, 'specimens': specimens, 'model_genes': model_genes, 'centers': centers, 'widths': widths}, code=CODE_DIGEST)
    stability['failure'] = stability.failure.fillna('')
    display(stability.query("failure != ''"))
    display(stability.query("failure == ''").groupby(['train_specimen', 'scheme', 'method', 'structure_id']).z.std().groupby(['train_specimen', 'scheme', 'method']).agg(['median', 'max']))
stability.to_csv(OUT / 'bootstrap_and_sparse_positions.csv', index=False)

## 6. Withheld markers and independent physical depth

Read physical information **after molecular fitting**. The pass-1 export must have reviewed Glomerulus labels. Validate each anchor's feature ID and centroid against current segmentation, then compute nearest-glomerulus distance and mean distance to the nearest three anchors per specimen. Never feed x/y or either distance into PCA, flow, or atlas fitting. These are cortical-depth proxies, not longitudinal PT ground truth. Report each specimen separately; no specimen pooling or biological-replicate p-values.

Plot excluded early/late markers afterward on frozen held-out positions. Sex-dimorphic S3 markers may disagree across controls. DPT and upstream labels still carry indirect marker information, so this is a diagnostic alongside randomized gene holdouts, not an independent proof.

In [ ]:
def physical_depth():
    if not GLOMERULUS_PATH.is_file():
        raise FileNotFoundError(f'Physical validation requires reviewed pass-1 glomeruli: {GLOMERULUS_PATH}')
    anchors = ad.read_h5ad(GLOMERULUS_PATH)
    needed = {'sample', 'coarse_class', 'feature_index', 'x_centroid', 'y_centroid'}
    if not needed.issubset(anchors.obs):
        raise ValueError(f'Glomerulus export missing {needed-set(anchors.obs)}')
    depth = np.full((len(X), 2), np.nan)
    for s in CONTROLS:
        take = (anchors.obs['sample'].astype(str)==s) & (anchors.obs.coarse_class.astype(str)=='Glomerulus')
        anchor_obs = anchors.obs.loc[take]
        if anchor_obs.empty:
            raise ValueError(f'{s}: no reviewed glomerular anchors')
        _read_and_validate_segmentation(DATA_ROOT, s, anchor_obs)
        anchor_xy = anchor_obs[['x_centroid', 'y_centroid']].to_numpy(float)
        tr = specimens == s
        distances, _ = cKDTree(anchor_xy).query(xy[tr], k=min(3, len(anchor_xy)))
        distances = np.asarray(distances).reshape(np.sum(tr), -1)
        depth[tr, 0], depth[tr, 1] = distances[:, 0], distances.mean(axis=1)
    return depth

depth = physical_depth()
physical_rows = []
for s in CONTROLS:
    take = specimens == s
    for j, proxy in enumerate(['nearest_glomerulus', 'mean_three_glomeruli']):
        physical_rows.append(dict(specimen=s, method='DPT', proxy=proxy, spearman=spearmanr(z_dpt[take], depth[take, j]).statistic))
if not latent_coordinates.empty:
    for key, group in latent_coordinates.groupby(['fold', 'train_specimen', 'initialization', 'anchor', 'method']):
        ix = pd.Index(ids).get_indexer(group.structure_id)
        for j, proxy in enumerate(['nearest_glomerulus', 'mean_three_glomeruli']):
            physical_rows.append(dict(specimen=specimens[ix[0]], fold=key[0], initialization=key[2], anchor=key[3], method=key[4], proxy=proxy, spearman=spearmanr(group.z, depth[ix, j]).statistic))
physical_scores = pd.DataFrame(physical_rows)
physical_scores.to_csv(OUT / 'physical_depth_correlations.csv', index=False)
display(physical_scores)
marker_curves = []
for s in CONTROLS:
    take = specimens == s
    coordinates = {'DPT': z_dpt[take]}
    if not latent_coordinates.empty:
        group = latent_coordinates.query("fold == 0 and initialization == 'DPT' and anchor == 'ordinal' and method == 'atlas_flow'")
        group = group[group.structure_id.isin(ids[take])].set_index('structure_id')
        if len(group) == np.sum(take):
            coordinates['atlas_flow'] = group.loc[ids[take], 'z'].to_numpy()
    for method, zz in coordinates.items():
        for g in np.flatnonzero(marker_mask):
            curve = predict_genes(X[take][:, [g]], zz, centers).ravel()
            for center, value in zip(centers, curve):
                marker_curves.append(dict(specimen=s, method=method, gene=genes[g], z=center, expression=value))
marker_curves = pd.DataFrame(marker_curves)
marker_curves.to_csv(OUT / 'withheld_marker_curves.csv', index=False)
for g in ['Slc5a2', 'Slc22a6', 'Slc7a13']:
    frame = marker_curves[marker_curves.gene.str.casefold() == g.casefold()]
    if not frame.empty:
        fig, ax = plt.subplots(figsize=(5, 3), layout='constrained')
        for (s, method), group in frame.groupby(['specimen', 'method']):
            ax.plot(group.z, group.expression, label=s+' '+method)
        ax.set(xlabel='Position (unitless gauge)', ylabel='Log-normalized expression', title=g+' — excluded molecular feature')
        ax.legend(fontsize=7)
        fig.savefig(OUT / (g+'_withheld.pdf'))
        plt.show()

## 7. Decision and optional frozen healthy-atlas AKI projection

No success is defined by agreement with DPT. A first latent gate requires all gene folds and both training directions to beat atlas-only and supplied DPT with the prespecified DPT/ordinal initialization. That is a descriptive screen. Review initialization agreement, sparse-window failures, uncertainty, withheld markers and physical-depth direction before extending the biological interpretation. No automatic gate can establish correct injured-tubule location without ground truth.

Enable `PSEUDOSPACE_15_AKI=1` only for a gated healthy-reference stress test. Load IR2A2/IR2A4 separately, apply the loader's recorded injury-cohort structure floor, and normalize using the **healthy measured gene denominator** before selecting reconstruction features. Pool the two healthy controls with exactly balanced empirical clouds at every fitted window (refuse a sparse specimen). Freeze the healthy scaler/PCA, atlas, and flow; injury never updates the healthy axis. Report posterior position, entropy, and negative healthy mixture log-density as an out-of-distribution score. This score is not a calibrated injury severity or gene-specific mechanism.

In [ ]:
latent_gate = False
if not latent_scores.empty:
    selected = latent_scores.query("initialization == 'DPT' and anchor == 'ordinal'")
    paired = selected.pivot(index=['fold', 'train_specimen'], columns='method', values='gene_mse')
    dpt_score = latent_scores.query("method == 'DPT'").set_index(['fold', 'train_specimen']).gene_mse
    latent_gate = bool(len(paired) == N_GENE_FOLDS*len(CONTROLS) and {'atlas', 'atlas_flow'}.issubset(paired) and (paired.atlas_flow < paired.atlas).all() and (paired.atlas_flow < dpt_score.reindex(paired.index)).all())
print('Descriptive latent gene-prediction screen:', latent_gate)
(OUT / 'latent_gate.json').write_text(json.dumps({'pass': latent_gate, 'rule': 'all gene folds and specimen directions: DPT-initialized ordinal atlas_flow beats atlas and supplied DPT', 'further_review_required': ['initialization', 'bootstrap', 'sparse_sampling', 'markers', 'physical_depth']}, indent=2))
healthy_result, healthy_transform = None, None
if latent_gate and (RUN_AKI or RUN_TF):
    H, _, healthy_transform = representation(X[:, model_genes], X[:0, model_genes], N_COMPONENTS, SEED)
    healthy_result = latent_fit(H, z_dpt, labels, centers, widths['overlapping'], flow_weight=.5, iterations=ITERATIONS, min_count=MIN_COUNT, max_points=MAX_POINTS, seed=SEED, ridge=RIDGE, epsilon=EPSILON, specimens=specimens)
if healthy_result is not None:
    np.savez_compressed(OUT / 'healthy_training_posterior.npz', structure_id=ids, centers=centers, posterior=healthy_result['posterior'], z=healthy_result['z'])
if RUN_AKI and healthy_result is not None:
    injured = load_repeated_mouse_inputs(DATA_ROOT, samples=('IR2A2', 'IR2A4'))
    if not set(genes).issubset(injured.var_names):
        raise ValueError('AKI lacks genes used in healthy normalization denominator')
    C = injured.layers['counts'][:, injured.var_names.get_indexer(genes)]
    C = C.toarray().astype(float) if sparse.issparse(C) else np.asarray(C, float)
    L = np.log1p(1e4*C / C.sum(axis=1, keepdims=True))
    I = healthy_transform['pca'].transform(healthy_transform['scaler'].transform(L[:, model_genes]))
    zz, probability, entropy = posterior(healthy_result['atlas'], I)
    atlas = healthy_result['atlas']
    lp = np.column_stack([-.5*np.sum(np.log(2*np.pi*v)+(I-m)**2/v, axis=1) for m,v in zip(atlas['means'],atlas['variances'])])
    abnormality = -logsumexp(lp-np.log(len(centers)), axis=1)
    injury_projection = injured.obs[['structure_id', 'sample', 'segment_class']].copy()
    injury_projection['z_healthy_atlas'] = zz
    injury_projection['posterior_entropy'] = entropy
    injury_projection['negative_log_healthy_density'] = abnormality
    injury_projection.to_csv(OUT / 'aki_frozen_healthy_projection.csv', index=False)
    np.savez_compressed(OUT / 'aki_position_posterior.npz', structure_id=injured.obs.structure_id.to_numpy(str), centers=centers, posterior=probability)
    (OUT / 'aki_input_manifest.json').write_text(json.dumps(dict(injured.uns['repeated_mouse_input_manifest']), indent=2))
    display(injury_projection.groupby('sample')[['z_healthy_atlas', 'posterior_entropy', 'negative_log_healthy_density']].agg(['median', 'min', 'max']))
else:
    print('AKI extension disabled or latent gate failed')

## 8. Optional TF Jacobian feasibility, after validation

Enable `PSEUDOSPACE_15_TF=1` and provide `PSEUDOSPACE_15_TF_LIST`, a curated mouse TF CSV containing `gene`. Only modeling genes can be perturbed. A linear PCA representation permits a chain-rule derivative of the fitted low-dimensional velocity with respect to log-normalized input expression. Report $\|\partial v/\partial x_g\|$ across $z$, not a gene-output Jacobian in full gene space. For this affine model the derivative depends on position but not state. It measures fitted transition sensitivity, **not causality**, regulator activity, or an intervention effect. Geneformer candidate prioritization can be supplied later as an external panel; it never constructs this coordinate.

This branch also requires a manual scientific review flag, `PSEUDOSPACE_15_VALIDATION_REVIEWED=1`, because gene-prediction gates alone do not demonstrate physical validity or stable latent ordering. This is a run configuration, not an approval step for creating the notebook.

In [ ]:
if RUN_TF and healthy_result is not None and os.environ.get('PSEUDOSPACE_15_VALIDATION_REVIEWED', '0') == '1':
    tf_path = os.environ.get('PSEUDOSPACE_15_TF_LIST')
    if not tf_path:
        raise ValueError('Provide a curated mouse TF CSV via PSEUDOSPACE_15_TF_LIST')
    tf_table = pd.read_csv(tf_path)
    if 'gene' not in tf_table or tf_table.gene.isna().any() or tf_table.gene.duplicated().any():
        raise ValueError('TF CSV needs unique nonmissing gene values')
    tf_set = set(tf_table.gene.astype(str).str.casefold())
    model = healthy_result['atlas']['flow_model']
    d = model['n_features']
    coef = model['coef']
    derivatives = healthy_transform['pca'].components_.T / healthy_transform['scaler'].scale_[:, None]
    influence_rows = []
    for j, g in enumerate(genes[model_genes]):
        if g.casefold() in tf_set:
            for center in centers:
                dv = derivatives[j] @ (coef[3:3+d] + center*coef[3+d:3+2*d])
                influence_rows.append(dict(gene=g, z=center, velocity_derivative_norm=float(np.linalg.norm(dv))))
    influence = pd.DataFrame(influence_rows)
    influence.to_csv(OUT / 'tf_velocity_sensitivity_feasibility.csv', index=False)
    (OUT / 'tf_panel_manifest.json').write_text(json.dumps({'source': str(tf_path), 'sha256': hashlib.sha256(Path(tf_path).read_bytes()).hexdigest(), 'interpretation': 'affine low-dimensional velocity sensitivity, not causality'}, indent=2))
    display(influence)
else:
    print('TF feasibility disabled, gate failed, or validation review not recorded')

## Interpretation and next decision

If transport fails in either specimen direction or non-overlap sensitivity, stop this version before latent fitting. If transport passes but latent flow does not improve unseen-gene prediction beyond atlas-only, probability flow has not earned its added complexity. Review uncertainty, sampling support, initialization dependence, physical-depth sensitivity and excluded marker patterns alongside every score; smoother plots and correlation with DPT are insufficient.

Even a successful pilot supports a reproducible molecular distributional path, not a unique metric spatial coordinate or mechanistic molecular dynamics. The next experiment would strengthen conditional density modeling, restricted specimen effects, and explicit held-out pushforward consistency, with new independent healthy specimens. The present two-control design cannot confirm superiority. Outputs are private analysis artifacts under the configured results root; the committed notebook stays output-free. No execution results are asserted by this source notebook.